In [1]:
from kafka import KafkaConsumer
from kafka import KafkaProducer
import json
import matplotlib.pyplot as plt
import time
import joblib
import numpy as np

In [2]:
model = joblib.load("../aeration_classifier.joblib")

kafka_bootstrap_servers = ["localhost:9092"]
kafka_topic_in = "water_quality"
kafka_topic_out = "water_quality_predict"
GROUP_ID = "water_quality_predict"

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.4.0 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.4.0 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [3]:
model

RandomForestClassifier(n_estimators=10, random_state=42)

In [4]:


def initialize_consumer():
    return KafkaConsumer(
        kafka_topic_in,
        bootstrap_servers=kafka_bootstrap_servers,
        key_deserializer=lambda k: k.decode("utf-8") if k else "unknown_sensor",
        value_deserializer=lambda m: json.loads(m.decode("utf-8")),
        auto_offset_reset="latest",
        enable_auto_commit=True,
        group_id=GROUP_ID,
    )



In [5]:
PROCESSOR_ID = "ML_1"
def predict(consumer, producer):
    try:
        for msg in consumer:
            sensor_key = msg.key
            sensor_data = msg.value
            print(f"Received: {sensor_data} Key: {sensor_key}")

            X = [[
                sensor_data["water_temperature"],
                sensor_data["ph_level"],
                sensor_data["turbidity"],
                sensor_data["dissolved_oxygen"]
            ]]

            pred = int(model.predict(X)[0])
            proba = model.predict_proba(X)[0].tolist()
            timestp = sensor_data["timestamp"]

            payload = {
                "sensor_id": sensor_key,
                "timestamp": timestp,
                "prediction": pred,
                "proba_class_0": proba[0],
                "proba_class_1": proba[1]
            }

            producer.send(kafka_topic_out, value=payload, key=sensor_key)
            print(f"[ML -> {PROCESSOR_ID}] key={sensor_key} pred={pred} proba={proba} timestamp={timestp}")
            break
    except KeyboardInterrupt:
        print("Stopped.")

In [6]:
consumer = initialize_consumer()

# Create Kafka producer
producer = KafkaProducer(
    bootstrap_servers=kafka_bootstrap_servers,
    key_serializer=lambda k: k.encode("utf-8"),
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)


In [ ]:
try:
    while True:
        predict(consumer, producer)
except KeyboardInterrupt:
    print("Stopped.")
    consumer.close()
    producer.close()

Stopped.
Received: {'sensor_id': '1', 'timestamp': 1772366487, 'water_temperature': 26.21204077881317, 'ph_level': 8.130915953811193, 'turbidity': 44.54, 'dissolved_oxygen': 8.92} Key: 1
[ML -> ML_1] key=1 pred=0 proba=[1.0, 0.0] timestamp=1772366487
Received: {'sensor_id': '3', 'timestamp': 1772366487, 'water_temperature': 26.854858499194613, 'ph_level': 8.219790380398328, 'turbidity': 24.32, 'dissolved_oxygen': 7.63} Key: 3
[ML -> ML_1] key=3 pred=1 proba=[0.0, 1.0] timestamp=1772366487
Received: {'sensor_id': '4', 'timestamp': 1772366487, 'water_temperature': 27.92035438044675, 'ph_level': 8.242365660014471, 'turbidity': 6.35, 'dissolved_oxygen': 10.01} Key: 4
[ML -> ML_1] key=4 pred=0 proba=[1.0, 0.0] timestamp=1772366487
Received: {'sensor_id': '1', 'timestamp': 1772366488, 'water_temperature': 26.32092673094903, 'ph_level': 8.457553947384698, 'turbidity': 14.98, 'dissolved_oxygen': 9.28} Key: 1
[ML -> ML_1] key=1 pred=0 proba=[1.0, 0.0] timestamp=1772366488
Received: {'sensor_id'